# bus_30min_pipeline

버스 30분 단위 일별 CSV를 월별 CSV로 집계하는 파이프라인입니다.

In [ ]:
import argparse
from pathlib import Path
from datetime import datetime
import zipfile

import pandas as pd


REQUIRED_OUTPUT_COLUMNS = [
    "YEAR",
    "MONTH",
    "DAY",
    "HOUR",
    "HALF_HOUR",
    "STATION_ID",
    "STATION_NM",
    "GETON_CNT",
    "GETOFF_CNT",
]


# Allow common source-column variants but force final schema to requested names.
COLUMN_SYNONYMS = {
    "YEAR": ["YEAR", "L (YEAR)", "년(YEAR)", "년 (YEAR)"],
    "MONTH": ["MONTH", "(MONTH)", "월(MONTH)", "월 (MONTH)"],
    "DAY": ["DAY", "(DAY)", "일(DAY)", "일 (DAY)"],
    "HOUR": ["HOUR", "시간(HOUR)", "시간 (HOUR)"],
    "HALF_HOUR": ["HALF_HOUR", "분_30분 단위 (HALF_HOUR)", "분_30분단위(HALF_HOUR)"],
    "STATION_ID": ["STATION_ID", "ID(STATION_ID)", "ID (STATION_ID)"],
    "STATION_NM": ["STATION_NM", "STATIONM", "STATION NM", "O (STATION_NM)", "정류장명(STATION_NM)"],
    "GETON_CNT": ["GETON_CNT", "승하차인원(GETON CNT)", "승차인원(GETON_CNT)", "승차인원(GETON CNT)"],
    "GETOFF_CNT": ["GETOFF_CNT", "하차인원(GETOFF_CNT)", "하차인원(GETOFF CNT)"],
}


def normalize_col_name(col: str) -> str:
    return str(col).strip().replace("\ufeff", "")


def harmonize_columns(df: pd.DataFrame) -> pd.DataFrame:
    src_cols = {normalize_col_name(c): c for c in df.columns}
    rename_map = {}

    for target, candidates in COLUMN_SYNONYMS.items():
        found = None
        for c in candidates:
            c_norm = normalize_col_name(c)
            if c_norm in src_cols:
                found = src_cols[c_norm]
                break

        if found is not None:
            rename_map[found] = target

    df = df.rename(columns=rename_map)
    missing = [c for c in REQUIRED_OUTPUT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return df[REQUIRED_OUTPUT_COLUMNS].copy()


def read_daily_csv(file_path: Path, sep: str = "|") -> pd.DataFrame:
    last_error = None

    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(file_path, engine="python", dtype=str, encoding=encoding, sep=sep)
            break
        except UnicodeDecodeError as e:
            last_error = e
    else:
        raise last_error

    df = harmonize_columns(df)

    for c in ["GETON_CNT", "GETOFF_CNT"]:
        df[c] = (
            df[c].astype(str)
            .str.replace("'", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    return df


def aggregate_monthly(month_df: pd.DataFrame) -> pd.DataFrame:
    # Monthly aggregation at station + 30-minute slot level (DAY removed).
    group_cols = ["YEAR", "MONTH", "HOUR", "HALF_HOUR", "STATION_ID", "STATION_NM"]
    out = (
        month_df.groupby(group_cols, dropna=False, as_index=False)[["GETON_CNT", "GETOFF_CNT"]]
        .sum()
        .sort_values(group_cols)
    )
    return out


def zip_output_folder(output_dir: Path) -> Path:
    ts = datetime.now().strftime("%m%d_%H%M%S")
    zip_path = output_dir.parent / f"monthly_aggregates_{ts}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in output_dir.glob("*.csv"):
            zf.write(file, arcname=file.name)

    return zip_path


def discover_month_folders(input_root: Path) -> list[Path]:
    # Case A: E:/2022/202201 ...
    direct_months = [
        p for p in input_root.iterdir()
        if p.is_dir() and len(p.name) == 6 and p.name.isdigit()
    ]
    if direct_months:
        return sorted(direct_months)

    # Case B: E:/2022/202201 ... where input_root is E:/
    year_dirs = [
        p for p in input_root.iterdir()
        if p.is_dir() and len(p.name) == 4 and p.name.isdigit()
    ]

    nested_months = []
    for y in year_dirs:
        nested_months.extend(
            [p for p in y.iterdir() if p.is_dir() and len(p.name) == 6 and p.name.isdigit()]
        )

    return sorted(nested_months)


def run_pipeline(input_root: Path, output_dir: Path, sep: str = "|") -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    month_dirs = discover_month_folders(input_root)
    if not month_dirs:
        raise FileNotFoundError(
            f"No month folders found under: {input_root}. Expected folders like 202201, 202202, ..."
        )

    written_files = []

    for month_dir in month_dirs:
        # Handle both:
        # 1) E:/2022/202201/TBDM...
        # 2) E:/2022/202201/202201/TBDM...
        nested_same_month = month_dir / month_dir.name
        data_dir = nested_same_month if nested_same_month.is_dir() else month_dir

        daily_files = sorted(data_dir.glob("TBDM_TRANSIT_STAT_BUS_*.csv"))
        if not daily_files:
            daily_files = sorted(data_dir.glob("*.csv"))

        if not daily_files:
            print(f"[WARN] No daily csv found in {data_dir}")
            continue

        monthly_frames = []
        for f in daily_files:
            try:
                monthly_frames.append(read_daily_csv(f, sep=sep))
            except Exception as e:
                print(f"[WARN] Skip file due to parse/schema issue: {f} -> {e}")

        if not monthly_frames:
            print(f"[WARN] No valid daily file in {data_dir}")
            continue

        month_df = pd.concat(monthly_frames, ignore_index=True)
        monthly_agg = aggregate_monthly(month_df)

        yy = month_dir.name[:4]
        mm = month_dir.name[4:6]
        out_file = output_dir / f"bus_30min_monthly_{yy}_{mm}.csv"
        monthly_agg.to_csv(out_file, index=False, encoding="utf-8-sig")
        written_files.append(out_file)
        print(f"[OK] {out_file} rows={len(monthly_agg):,}")

    if not written_files:
        raise RuntimeError("No monthly output was generated. Check folder structure/columns/separator.")

    zip_path = zip_output_folder(output_dir)
    print("\n=== Pipeline Completed ===")
    print(f"Monthly files: {len(written_files)}")
    print(f"Zip: {zip_path}")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Subway 30-min daily CSV -> monthly aggregate CSV pipeline"
    )
    parser.add_argument(
        "--input-root",
        required=True,
        help="Root folder (E drive) containing year/month/day CSV hierarchy, e.g. E:/subway_data",
    )
    parser.add_argument(
        "--output-dir",
        required=True,
        help="Output folder for monthly CSV files",
    )
    parser.add_argument(
        "--sep",
        default="|",
        help="CSV separator (default: |)",
    )

    args = parser.parse_args()
    run_pipeline(
        input_root=Path(args.input_root),
        output_dir=Path(args.output_dir),
        sep=args.sep,
    )

## 실행 셀

실서버 경로에 맞게 수정해서 실행하세요.

In [ ]:
# 실행 셀 (실서버 경로에 맞게 수정해서 실행)
input_root = Path(r"E:\janghogeun\버스파이프라인")
output_dir = Path(r"E:\janghogeun\버스파이프라인_monthly_out")

run_pipeline(input_root=input_root, output_dir=output_dir)

# 빅데캠 내부 pc 수정 부분

In [ ]:
def read_daily_csv(file_path: Path, sep: str = ",") -> pd.DataFrame:
    last_error = None

    for encoding in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            rows = []
            with open(file_path, "r", encoding=encoding, newline="") as f:
                reader = csv.reader(f, delimiter=sep)
                header = next(reader)

                for line_no, row in enumerate(reader, start=2):
                    if len(row) == 9:
                        fixed = row
                    elif len(row) > 9:
                        # YEAR, MONTH, DAY, HOUR, HALF_HOUR, STATION_ID = first 6
                        # GETON_CNT, GETOFF_CNT = last 2
                        # Everything between belongs to STATION_NM
                        fixed = row[:6] + [sep.join(row[6:-2])] + row[-2:]
                    else:
                        print(f"[WARN] Skip short row: {file_path} line={line_no} fields={len(row)}")
                        continue

                    rows.append(fixed)

            df = pd.DataFrame(rows, columns=REQUIRED_OUTPUT_COLUMNS)
            break

        except UnicodeDecodeError as e:
            last_error = e
    else:
        raise last_error

    for c in REQUIRED_OUTPUT_COLUMNS:
        df[c] = df[c].astype(str).str.strip().str.strip("`").str.strip("'").str.strip('"')

    for c in ["GETON_CNT", "GETOFF_CNT"]:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    return df

In [ ]:
def run_pipeline(input_root: Path, output_dir: Path, sep: str = ",") -> None:

In [ ]:
run_pipeline(input_root=input_root, output_dir=output_dir, sep=",")